# Section 5 — Runge–Kutta Methods: RK2 and RK3

[← Taylor Methods](04_taylor_methods.ipynb) · [Module 3](README.md)

Runge–Kutta methods obtain higher-order accuracy **without explicitly differentiating** \(f(t,y)\).  
Instead, they evaluate the slope \(f\) at several carefully chosen points during each step.

This notebook follows the progression used in **Leon Q. Brin, _Tea Time Numerical Analysis_, Chapter 6**, and the standard Runge–Kutta notation used by **Burden & Faires**.

### Learning goals
- implement two second-order Runge–Kutta methods;
- implement Heun's three-stage third-order method;
- compare the methods on the same IVP;
- verify the expected global convergence orders numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    return t*np.exp(3*t) - 2*y

def exact(t):
    return np.exp(3*t)*(t/5 - 1/25) + np.exp(-2*t)/25

t0, y0, tf = 0.0, 0.0, 1.0

## The test problem

We use the same IVP as in the Taylor-method notebook:

\[
y' = t e^{3t}-2y,\qquad y(0)=0,
\]

with exact solution

\[
y(t)=e^{3t}\left(\frac{t}{5}-\frac{1}{25}\right)+\frac{e^{-2t}}{25}.
\]

At \(t=1\), \(y(1)\approx 3.219099\).

## Improved Euler / Heun RK2

The first RK2 variant uses the slope at the start and a predicted slope at the end:

\[
\begin{aligned}
k_1 &= f(t_n,w_n),\\
k_2 &= f(t_n+h,w_n+h k_1),\\
w_{n+1} &= w_n+\frac{h}{2}(k_1+k_2).
\end{aligned}
\]

This is a second-order method: the global error is \(O(h^2)\).

In [ ]:
def rk2_improved_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h, y + h*k1)
    return y + 0.5*h*(k1 + k2)

def rk2_midpoint_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + 0.5*h, y + 0.5*h*k1)
    return y + h*k2

def rk3_heun_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/3, y + (h/3)*k1)
    k3 = f(t + 2*h/3, y + (2*h/3)*k2)
    return y + (h/4)*(k1 + 3*k3)

def fixed_step_solver(step_function, f, t0, y0, tf, h):
    n = round((tf - t0)/h)
    t = np.empty(n+1)
    y = np.empty(n+1)
    t[0], y[0] = t0, y0
    for i in range(n):
        y[i+1] = step_function(f, t[i], y[i], h)
        t[i+1] = t[i] + h
    return t, y

## Midpoint RK2

The midpoint method first predicts a value at the midpoint and then uses the midpoint slope for the full step:

\[
\begin{aligned}
k_1 &= f(t_n,w_n),\\
k_2 &= f\left(t_n+\frac h2,w_n+\frac h2 k_1\right),\\
w_{n+1} &= w_n+h k_2.
\end{aligned}
\]

It is also order 2, but it has a different error constant from Improved Euler.

## Heun's three-stage RK3

The third-order method used in the course notes and in Brin's RK discussion is

\[
\begin{aligned}
k_1 &= f(t_n,w_n),\\
k_2 &= f\left(t_n+\frac h3,w_n+\frac h3k_1\right),\\
k_3 &= f\left(t_n+\frac{2h}{3},w_n+\frac{2h}{3}k_2\right),\\
w_{n+1} &= w_n+\frac h4(k_1+3k_3).
\end{aligned}
\]

Its global error is \(O(h^3)\).

In [ ]:
h = 0.5
methods = {
    "Improved Euler (RK2)": rk2_improved_step,
    "Midpoint (RK2)": rk2_midpoint_step,
    "Heun (RK3)": rk3_heun_step,
}

print(f"Exact y(1) = {exact(1.0):.9f}\n")
print(f"{'Method':24s} {'Approximation':>14s} {'Abs. error':>14s}")
print("-"*56)

solutions = {}
for name, step in methods.items():
    t, y = fixed_step_solver(step, f, t0, y0, tf, h)
    solutions[name] = (t, y)
    print(f"{name:24s} {y[-1]:14.9f} {abs(y[-1]-exact(tf)):14.6e}")

In [ ]:
t_fine = np.linspace(t0, tf, 400)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label="Exact solution")
for name, (t, y) in solutions.items():
    ax.plot(t, y, marker="o", label=name)
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("RK2 and RK3 with h = 0.5")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Convergence-order experiment

If a method has global error \(E(h)\approx C h^p\), then halving the step size gives

\[
\frac{E(h)}{E(h/2)}\approx 2^p.
\]

Therefore an observed order can be estimated by

\[
p \approx \log_2\!\left(\frac{E(h)}{E(h/2)}\right).
\]

In [ ]:
step_sizes = np.array([0.25, 0.125, 0.0625, 0.03125])

for name, step in methods.items():
    errors = []
    for h in step_sizes:
        _, y = fixed_step_solver(step, f, t0, y0, tf, h)
        errors.append(abs(y[-1] - exact(tf)))
    print("\n" + name)
    print(f"{'h':>10s} {'error':>14s} {'observed p':>12s}")
    previous = None
    for h, err in zip(step_sizes, errors):
        p = "--" if previous is None else f"{np.log2(previous/err):.4f}"
        print(f"{h:10.5f} {err:14.6e} {p:>12s}")
        previous = err

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for name, step in methods.items():
    errors = []
    for h in step_sizes:
        _, y = fixed_step_solver(step, f, t0, y0, tf, h)
        errors.append(abs(y[-1] - exact(tf)))
    ax.loglog(step_sizes, errors, marker="o", label=name)
ax.set_xlabel("step size h")
ax.set_ylabel("absolute error at t=1")
ax.set_title("Convergence of RK2 and RK3")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.show()

## Takeaway

Using additional slope information raises the order without requiring \(f_t\), \(f_y\), or higher derivatives.

- Improved Euler: order 2, two \(f\)-evaluations per step.
- Midpoint: order 2, two \(f\)-evaluations per step.
- Heun RK3: order 3, three \(f\)-evaluations per step.

Next we move to the classical fourth-order Runge–Kutta method.

[Next: RK4 →](06_rk4.ipynb)